In [7]:
from ase.io import read, write
from ase import units
import numpy as np
from imolcraft.trainer import ThermodynamicTrainer
from imolcraft.trainer.dmff_utils import parser_dmffyaml
from imolcraft.trainer.loss import loss_thermodynamicperturbation
from functools import partial

In [8]:
params = parser_dmffyaml("dmff.yml")
params

{'sampling': {'init_structure': 'merged_supercell_bonds.pdb',
  'ensemble': 'anisonpt',
  'dt_fs': 1.0,
  'rcut_nm': 0.8,
  'nonbondedmethod': 'LJPME',
  'temperature_K': 233.15,
  'pressure_bar': 1.0,
  'anneal_steps': 1,
  'anneal_Tmax': 400.0,
  'anneal_totalsteps': 0,
  'relax_steps': 20000,
  'prod_steps': 200000,
  'nstxout': 1000,
  'neff': 30,
  'dispcorr': False,
  'anneal_totaltime': 0},
 'targets': {'La_A': {'weight': 0.1, 'gt': 24.188},
  'Lb_A': {'weight': 0.1, 'gt': 24.188},
  'Lc_A': {'weight': 0.1, 'gt': 20.1076},
  'rdf': {'Li-O': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'O',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_O.txt'},
   'Li-N': {'weight': 1.0,
    'elem1': 'Li',
    'elem2': 'N',
    'rcut12_A': 8.0,
    'gt': 'reference_rdf_Li_N.txt'}},
  'adf': {'C-N-Li': {'weight': 1.0,
    'elem1': 'C',
    'elem2': 'N',
    'elem3': 'Li',
    'rcut12_A': 1.4,
    'rcut23_A': 2.9,
    'gt': 'reference_adf_CNLi.txt'}}}}

In [9]:
# for test 
# params["sampling"]["relax_steps"] = 1000
# params["sampling"]["prod_steps"] = 2000

In [10]:
lossfn = partial(loss_thermodynamicperturbation, losstype_distribfn="wrightfactor")

In [12]:
lr_prod = 5e-4

trainer = ThermodynamicTrainer(
    ffxml_list=["SN.xml", "FSA.xml", "Li.xml"],
    nums_ffxml=[2,1,1],
    pdbfile="supercell_bonds.pdb",
    loss_fn=lossfn,
    sampling_params=params["sampling"],
    target_params=params["targets"],
    opt_fftypes=["NonbondedForce/charge", 'VirtualSite/vsite_w2_type_2'],
    label="2rdf_1adf",
    device="OpenCL", # CPU or OpenCL or CUDA
    lr=lr_prod,
)

In [13]:
trainer.ffparams

{'HarmonicBondForce': {'length': Array([0.1153, 0.1467, 0.1538, 0.1097, 0.1612, 0.1453, 0.1719], dtype=float64),
  'k': Array([746040.672, 247575.648, 194572.736, 314569.856, 335431.28 ,
         571559.504, 237098.912], dtype=float64)},
 'HarmonicAngleForce': {'angle': Array([3.11680898, 1.94970731, 1.90956473, 1.91637152, 1.87762521,
         2.08881005, 1.85947379, 1.69183491, 2.09526777, 1.86785137],      dtype=float64),
  'k': Array([ 612.671488,  554.597568,  409.019472,  391.756288,  326.01728 ,
          530.556304, 1089.488496,  567.551232, 1072.7776  , 1026.703392],      dtype=float64)},
 'PeriodicTorsionForce': {'proper_phase': Array([3.14159265, 3.14159265, 0.        , 0.        , 0.        ,
         2.16387219, 2.27151958], dtype=float64),
  'proper_k': Array([ 0.        ,  0.        ,  0.65084444,  0.65084444,  0.50208   ,
         45.65444   , 35.24682804], dtype=float64),
  'improper_phase': Array([], shape=(0,), dtype=float64),
  'improper_k': Array([], shape=(0,), dt

In [14]:
trainer.ffparams["NonbondedForce"]["charge"].shape

(22,)

In [15]:
from imolcraft.trainer.dmff_utils import neutralize

def grad_modify(grads):
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[12].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[13].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[14].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[15].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[16].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[17].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[18].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[19].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[20].set(0.0)
    grads["NonbondedForce"]["charge"] = grads["NonbondedForce"]["charge"].at[21].set(0.0)
    vsite_weight = (grads["VirtualSite"]["vsite_w2_type_2"][0]+grads["VirtualSite"]["vsite_w2_type_2"][1])/2
    grads["VirtualSite"]["vsite_w2_type_2"] = grads["VirtualSite"]["vsite_w2_type_2"].at[0].set(vsite_weight)
    grads["VirtualSite"]["vsite_w2_type_2"] = grads["VirtualSite"]["vsite_w2_type_2"].at[1].set(vsite_weight)
    return grads

def ffparams_modify(ffparams):
    ffparams = neutralize(ffparams, trainer.natoms_list, target_lists=[[0,1,2,3,4,5,6,7,8,9,10,11]], target_charges=[0.0])
    return ffparams

In [16]:
trainer.add_modifyfn("after_grad", grad_modify)
trainer.add_modifyfn("after_update", ffparams_modify)

In [ ]:
trainer.setup()

In [ ]:
trainer.fit(100, 2)